<a href="https://colab.research.google.com/github/flathfk/Bootcamp_Study/blob/main/260402.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TRIE

### 1. Trie의 정의와 목적

**정의**

Trie(트라이)는 문자열을 효율적으로 저장하고 탐색하기 위해 설계된 트리 기반 자료구조로서 각 노드가 하나의 문자를 나타내고 루트부터 특정 노드까지의 경로가 하나의 문자열을 의미하는 구조이며 공통 접두어(prefix)를 공유하는 문자열들을 하나의 경로로 압축하여 저장하는 특징을 가지며 검색 시 문자열을 문자 단위로 분해하여 트리를 따라가면서 탐색하는 방식으로 동작한다.

**목적**

- **문자열 탐색 최적화:** 문자열 길이를 기준으로 O(L) 시간에 검색 가능하도록 한다.
- **접두어 기반 검색:** 자동완성, 추천 시스템과 같이 prefix 기반 탐색을 효율적으로 처리한다.
- **중복 데이터 제거:** 공통 접두어를 공유하여 메모리를 절약한다.
- **빠른 존재 여부 확인:** 문자열 존재 여부를 빠르게 판단할 수 있다.

### 2. Trie와 알고리즘의 관계

> **"Trie는 문자열 탐색 알고리즘의 핵심 인프라이다"**
>
- **상호작용:** Trie는 문자열을 구조적으로 저장하고 알고리즘은 이를 기반으로 빠른 탐색 및 추천을 수행한다.
- **비유:** Trie는 “단어 사전의 색인 구조”이고 알고리즘은 “해당 색인을 따라가며 단어를 찾는 과정”이다.
- **의존성:** 자동완성, 사전 검색, 문자열 필터링 알고리즘은 Trie 구조를 기반으로 성능이 결정된다.

### 3. 시간 복잡도 (Trie 기준)

Trie의 연산은 문자열 길이 L에 의해 결정되며 데이터 개수와 무관하게 일정한 탐색 성능을 가진다.

| **연산** | **시간 복잡도** | **특징** | **설명** |
| --- | --- | --- | --- |
| **Insert** | O(L) | 문자 단위 삽입 | 문자열 길이만큼 노드 생성 |
| **Search** | O(L) | 문자 단위 탐색 | 경로 존재 여부 확인 |
| **StartsWith** | O(L) | 접두어 탐색 | prefix 기반 검색 |
| **Delete** | O(L) | 조건적 삭제 | 불필요 노드 제거 |

### 4. Trie의 구조와 메모리 표현

Trie는 트리 구조이며 각 노드는 자식 노드를 참조하는 구조로 구성된다.

- 각 노드는 `children` 객체를 통해 다음 문자로 연결된다.
- 문자열의 끝은 `isEnd` 플래그로 구분된다.
- 실제 메모리에서는 객체(Reference Type) 기반으로 Heap 영역에 저장된다.

### 5. 핵심 알고리즘

**Insert**

문자열을 한 글자씩 순회하면서 존재하지 않는 노드를 생성하고 마지막 문자에 도달하면 종료 표시를 한다.

**Search**

문자열을 한 글자씩 따라가며 경로가 존재하는지 확인하고 마지막 노드의 종료 여부를 확인한다.

**Prefix Search**

문자열 전체가 아닌 특정 접두어까지만 탐색하여 해당 prefix로 시작하는 문자열이 존재하는지 확인한다.

In [1]:
%%writefile ex1.js

// [1] 트라이의 각 노드 정의
//     children: 다음 글자로 이어지는 자식 노드들 저장
//     isEnd: 이 노드에서 끝나는 단어가 있는지 표시
class TrieNode {
    constructor() {
        this.children = {}; // ex) { a: TrieNode, p: TrieNode }
        this.isEnd = false; // "app" 마지막 p노드면 true
    }
}

class Trie {
    // [2] 루트 노드 생성 -> 모든 단어의 시작점
    constructor() {
        this.root = new TrieNode();
    }

    // [3] 단어 삽입: 글자 하나씩 노드 타고 내려가며 경로 생성
    insert(word) {
        let node = this.root;
        for (const char of word) {
            // [3-1] 해당 글자의 자식 노드 없으면 새로 생성
            if (!node.children[char]) {
                node.children[char] = new TrieNode();
            }
            // [3-2] 해당 글자 노드로 이동
            node = node.children[char];
        }
        // [3-3] 마지막 글자 노드에 단어 끝 표시
        node.isEnd = true;
    }

    // [4] 단어 탐색: 글자 하나씩 따라가다가 isEnd 확인
    search(word) {
        let node = this.root;
        for (const char of word) {
            // [4-1] 경로 중간에 노드 없으면 단어 없음
            if (!node.children[char]) return false;
            node = node.children[char];
        }
        // [4-2] 끝까지 도달했어도 isEnd가 false면 단어 아님
        //       ex) "appl" 탐색 시 노드는 있지만 isEnd=false -> false
        return node.isEnd;
    }

    // [5] 접두사 탐색: 해당 접두사로 시작하는 단어가 있는지 확인
    startsWith(prefix) {
        let node = this.root;
        for (const char of prefix) {
            // [5-1] 경로 중간에 노드 없으면 해당 접두사 없음
            if (!node.children[char]) return false;
            node = node.children[char];
        }
        // [5-2] 끝까지 경로 있으면 접두사 존재 (isEnd 상관없음)
        return true;
    }
}

// [6] 특정 노드 아래의 모든 단어 개수를 재귀로 카운트
function countWords(node) {
    // [6-1] 현재 노드가 단어 끝이면 1로 시작
    let count = node.isEnd ? 1 : 0;
    for (const char in node.children) {
        // [6-2] 자식 노드들 재귀 탐색하며 단어 수 누적
        count += countWords(node.children[char]);
    }
    return count;
}

// [7] 특정 접두사로 시작하는 단어 개수 반환
function countPrefix(trie, prefix) {
    let node = trie.root;
    for (const char of prefix) {
        // [7-1] 접두사 경로 없으면 해당 접두사로 시작하는 단어 없음
        if (!node.children[char]) return 0;
        node = node.children[char];
    }
    // [7-2] 접두사 끝 노드부터 하위 단어 전부 카운트
    return countWords(node);
}

// [8] 테스트
const trie = new Trie();
trie.insert("apple");
trie.insert("app");
trie.insert("apex");

console.log(trie.search("apple"));    // -> true  ("apple" 존재)
console.log(trie.search("app"));      // -> true  ("app" 존재)
console.log(trie.search("appl"));     // -> false (isEnd=false)
console.log(trie.startsWith("app"));  // -> true  ("ap"로 시작하는 경로 존재)
console.log(countPrefix(trie, "ap")); // -> 3     (apple, app, apex)

Writing ex1.js


In [2]:
!node ex1.js

true
true
false
true
3




---



In [3]:
%%bash

node -v
npm -v

v20.19.0
10.8.2


In [4]:
%%bash

cd /content
mkdir tailwind
cd tailwind

In [5]:
%%bash

npm init -y

Wrote to /content/package.json:

{
  "name": "content",
  "version": "1.0.0",
  "main": "ex1.js",
  "scripts": {
    "test": "echo \"Error: no test specified\" && exit 1"
  },
  "keywords": [],
  "author": "",
  "license": "ISC",
  "description": ""
}





In [6]:
%%bash

npm install tailwindcss @tailwindcss/cli express


added 92 packages, and audited 93 packages in 11s

28 packages are looking for funding
  run `npm fund` for details

found 0 vulnerabilities


In [12]:
%%bash

mkdir -p /content/tailwind/templates
mkdir -p /content/tailwind/static/css

In [21]:
%%bash

cat <<EOF > input.css
@import "tailwindcss";
EOF

In [13]:
%%writefile /content/tailwind/templates/index.html

<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <title>데이터 대시보드</title>
    <link href="/static/css/output.css" rel="stylesheet">
</head>
<body class="bg-gray-100">

<div class="max-w-5xl mx-auto p-8">
    <h1 class="text-3xl font-bold mb-6">데이터 대시보드</h1>

    <div class="grid grid-cols-3 gap-6">
        <div class="bg-white p-6 rounded-xl shadow">
            <p>사용자 수</p>
            <p class="text-2xl font-bold">12,340</p>
        </div>

        <div class="bg-white p-6 rounded-xl shadow">
            <p>매출</p>
            <p class="text-2xl font-bold">₩8,200,000</p>
        </div>

        <div class="bg-white p-6 rounded-xl shadow">
            <p>전환율</p>
            <p class="text-2xl font-bold">23%</p>
        </div>
    </div>

</div>

</body>
</html>

Writing /content/tailwind/templates/index.html


In [22]:
%%bash

npx @tailwindcss/cli -i ./input.css -o ./static/css/output.css --content "./templates/**/*.html"

≈ tailwindcss v4.2.2

Done in 1s


In [15]:
%%writefile /content/tailwind/server.js

const express = require('express');
const path = require('path');

const app = express();
const PORT = 3000;

app.use('/static', express.static(path.join(__dirname, 'static')));

app.get('/', (req, res) => {
    res.sendFile(path.join(__dirname, 'templates', 'index.html'));
});

app.listen(PORT, () => {
    console.log('Server running on http://localhost:' + PORT);
});

Writing /content/tailwind/server.js


In [23]:
!npm install express && npm install -g cloudflared
!nohup node server.js > server.log 2>&1 &
!nohup cloudflared tunnel --url http://localhost:3000 > tunnel.log 2>&1 &
!tail -n 20 tunnel.log

⠙⠹⠸⠼⠴⠦
up to date, audited 96 packages in 842ms
⠦
⠦30 packages are looking for funding
⠦  run `npm fund` for details
⠦
found 0 vulnerabilities
⠦⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹
changed 1 package in 1s
⠹

In [24]:
!nohup node /content/tailwind/server.js > server.log 2>&1 &

In [34]:
!grep -o 'https://.*\.trycloudflare\.com' tunnel.log

https://output-boxed-allowing-providers.trycloudflare.com


In [33]:
!mkdir -p /content/tailwind/static/css
!cp /content/static/css/output.css /content/tailwind/static/css/output.css